# MMedRAG-MedGemma · Radiology Report Generation
**Pipeline**: OpenCLIP (top-3) → RadGraph-XL (entities) → MedGemma-4B DPO

## Setup checklist (do before running)
1. Accelerator → **GPU T4 × 2**
2. Internet → **ON**
3. Kaggle Secrets:
   - `HF_TOKEN` — from huggingface.co/settings/tokens (needs access to `google/medgemma-4b-it`)
   - `WANDB_API_KEY` — from wandb.ai/settings
4. Accept MedGemma licence at **huggingface.co/google/medgemma-4b-it**
5. Add your repo as a Kaggle Dataset input (or use the git clone cell)
6. Add IU X-Ray images as a Kaggle Dataset input

## 1 · GPU check

In [ ]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPUs    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory // 1024**3} GB')

assert torch.cuda.device_count() >= 2, 'Need 2× T4 — enable GPU T4×2 in notebook settings'

## 2 · Install dependencies

In [ ]:
import subprocess, sys, shutil

# Bootstrap uv (fast pip replacement — resolves deps in seconds)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
uv = shutil.which('uv') or f'{sys.prefix}/bin/uv'
print(f'uv: {uv}')

pkgs = [
    # Core ML stack
    'transformers==4.50.0',
    'tokenizers==0.21.0',
    'accelerate==0.34.2',
    'peft==0.13.2',
    'trl==0.12.1',
    'bitsandbytes>=0.45.0',   # 0.45+ supports triton 3.x
    'safetensors==0.4.5',
    'huggingface-hub>=0.26.5,<2.0',
    'datasets==2.21.0',       # trl 0.12.1 requires >=2.21
    # Medical
    'radgraph',
    # Vision / retrieval
    'timm==1.0.12',
    'einops>=0.7.0',
    'ftfy',
    # Evaluation
    'evaluate>=0.4.0',
    'rouge-score>=0.1.2',
    'sacrebleu>=2.3.0',
    # Utilities
    'wandb>=0.16.0',
    'gdown>=5.1.0',
]

subprocess.run(
    [uv, 'pip', 'install', '--system', '--no-cache'] + pkgs,
    check=True,
)

print('\nAll packages installed.')
print('>>> Restart the kernel now: Runtime → Restart runtime <<<')


## 3 · Repo + credentials

In [ ]:
w

## 4 · Paths — edit these for your datasets

In [ ]:
import os, sys, shutil, subprocess

# ── Secrets (reads Kaggle secrets directly so this cell is self-contained) ──
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ.setdefault('HF_TOKEN',      _s.get_secret('HF_TOKEN'))
    os.environ.setdefault('WANDB_API_KEY', _s.get_secret('WANDB_API_KEY'))
    print('Secrets loaded from Kaggle.')
except Exception as e:
    print(f'Kaggle secrets unavailable ({e}) — set HF_TOKEN / WANDB_API_KEY manually.')


WORK = '/kaggle/working'
REPO = f'{WORK}/MMedRAGGX'

if not os.path.isdir(REPO):
    subprocess.run(
        ['git', 'clone', 'https://github.com/malakiss/MMedRAGGX.git', REPO],
        check=True,
    )
    print('Cloned repo from GitHub')

# ─── Edit these to match your Kaggle Dataset input paths ─────────────────────

# IU X-Ray images (uploaded as a Kaggle Dataset)
IU_IMG_ROOT = '/kaggle/input/iu-xray/images'

# MIMIC-CXR images (optional — only needed for test inference)
MIMIC_IMG_ROOT = '/kaggle/input/mimic-cxr/files'

# Alignment JSON (already in the repo)
ALIGN_RAW      = f'{REPO}/data/training/alignment/radiology/radiology_report.json'
ALIGN_RADGRAPH = f'{REPO}/data/training/alignment/radiology/radiology_report_radgraph.json'

# Output paths
RETRIEVAL_OUT  = f'{WORK}/output_rag_radgraph.json'
CHECKPOINT_DIR = f'{WORK}/checkpoints/medgemma_dpo'
RESULTS_FILE   = f'{WORK}/results_medgemma.json'

# image_root remap: JSON has /home/wenhao/... paths baked in
IMAGE_ROOT_REMAP = (
    f'/home/wenhao/Datasets/med/rad/iu_xray/images:{IU_IMG_ROOT},'
    f'/home/wenhao/Datasets/med/rad/mimic-cxr-jpg/physionet.org/files/mimic-cxr-jpg/2.0.0/files:{MIMIC_IMG_ROOT}'
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('IU images    :', IU_IMG_ROOT, '→', os.path.isdir(IU_IMG_ROOT))
print('Alignment JSON:', os.path.isfile(ALIGN_RAW))

# ── CLIP checkpoint (fine-tuned, downloaded from Google Drive) ──────────────
CLIP_CKPT = f'{WORK}/clip_checkpoint.pt'
GDRIVE_FILE_ID = '15LgW7TunV6ELbqcEWhgFdMQwDubZHtaw'
if not os.path.isfile(CLIP_CKPT):
    import gdown
    print('Downloading CLIP checkpoint from Google Drive ...')
    gdown.download(id=GDRIVE_FILE_ID, output=CLIP_CKPT, quiet=False)
    print(f'Saved → {CLIP_CKPT}')
else:
    print(f'CLIP checkpoint already present: {CLIP_CKPT}')


## 5 · RadGraph preprocessing (one-time, ~10 min CPU)

In [ ]:
import os

if os.path.isfile(ALIGN_RADGRAPH):
    print(f'Already exists: {ALIGN_RADGRAPH} — skipping.')
else:
    result = subprocess.run([
        sys.executable, f'{REPO}/train/medgemma/preprocess_radgraph_alignment.py',
        '--input',      ALIGN_RAW,
        '--output',     ALIGN_RADGRAPH,
        '--model_type', 'radgraph-xl',
        '--batch_size', '32',
    ])
    assert result.returncode == 0, 'RadGraph preprocessing failed'

## 6 · Accelerate config for 2× T4 DDP

In [ ]:
import os

ACCEL_CFG_DIR  = os.path.expanduser('~/.cache/huggingface/accelerate')
ACCEL_CFG_FILE = f'{ACCEL_CFG_DIR}/default_config.yaml'
os.makedirs(ACCEL_CFG_DIR, exist_ok=True)

accel_yaml = """\
compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
downcast_bf16: 'no'
gpu_ids: all
machine_rank: 0
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
use_cpu: false
"""

with open(ACCEL_CFG_FILE, 'w') as f:
    f.write(accel_yaml)

print(f'Accelerate config written → {ACCEL_CFG_FILE}')

## 7 · MedGemma DPO training on 2× T4

In [ ]:
import subprocess, sys, os

train_args = [
    '--model_name_or_path',          'google/medgemma-4b-it',
    '--data_path',                   ALIGN_RADGRAPH,
    '--image_folder',                IU_IMG_ROOT,
    '--image_root_remap',            IMAGE_ROOT_REMAP,
    '--output_dir',                  CHECKPOINT_DIR,
    '--max_length',                  '1024',
    '--use_4bit',                    'True',
    '--lora_enable',                 'True',
    '--lora_r',                      '128',
    '--lora_alpha',                  '256',
    '--lora_dropout',                '0.05',
    '--beta',                        '0.1',
    '--num_train_epochs',            '3',
    '--per_device_train_batch_size', '1',
    '--gradient_accumulation_steps', '4',   # effective batch = 2 GPUs × 1 × 4 = 8
    '--learning_rate',               '1e-7',
    '--lr_scheduler_type',           'cosine',
    '--warmup_ratio',                '0.03',
    '--weight_decay',                '0.0',
    '--bf16',                        'True',
    '--gradient_checkpointing',      'True',
    '--logging_steps',               '10',
    '--save_steps',                  '100',
    '--save_total_limit',            '2',
    '--report_to',                   'wandb',
    '--run_name',                    'medgemma_dpo_radiology_2xT4',
    '--dataloader_num_workers',      '2',
    '--remove_unused_columns',       'False',
]

import shutil
accel_bin = shutil.which('accelerate') or f'{sys.prefix}/bin/accelerate'

cmd = [
    accel_bin, 'launch',
    '--num_processes', '2',
    '--mixed_precision', 'bf16',
    f'{REPO}/train/medgemma/train_dpo_medgemma.py',
] + train_args

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0,1'

result = subprocess.run(cmd, env=env)
print('\nTraining exit code:', result.returncode)
assert result.returncode == 0, 'Training failed — check output above'

## 8 · OpenCLIP top-3 retrieval + RadGraph (for inference)

In [ ]:
import subprocess, sys, os

# CLIP_CKPT is set in Cell 4 (downloaded from Google Drive)
# Use IU X-Ray val split as eval set (change to mimic_test.json for MIMIC)
EVAL_JSON = f'{REPO}/data/training/retriever/radiology/rad_val_iu.json'
result = subprocess.run([
    sys.executable, f'{REPO}/train/open_clip/src/retrieve_clip_radgraph.py',
    '--img_root',           IU_IMG_ROOT,
    '--train_json',         f'{REPO}/data/training/retriever/radiology/rad_iu.json',
    '--eval_json',          EVAL_JSON,
    '--model_name_or_path', 'hf-hub:thaottn/OpenCLIP-resnet50-CC12M',
    '--checkpoint_path',    CLIP_CKPT,
    '--output_path',        RETRIEVAL_OUT,
    '--fixed_k',            '3',
    '--radgraph_model',     'radgraph-xl',
    '--radgraph_batch',     '32',
], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
assert result.returncode == 0, 'Retrieval failed'
print(f'Retrieval output → {RETRIEVAL_OUT}')

## 9 · Inference

In [ ]:
import subprocess, sys, os

result = subprocess.run([
    sys.executable, f'{REPO}/train/medgemma/inference_medgemma.py',
    '--checkpoint',     CHECKPOINT_DIR,
    '--base_model',     'google/medgemma-4b-it',
    '--retrieved',      RETRIEVAL_OUT,
    '--img_root',       IU_IMG_ROOT,
    '--output',         RESULTS_FILE,
    '--max_new_tokens', '256',
    '--temperature',    '0.1',
    '--use_4bit',
], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})

assert result.returncode == 0, 'Inference failed'
print(f'Results → {RESULTS_FILE}')

## 10 · Evaluate (BLEU / ROUGE / RadGraph-F1)

In [ ]:
import json, evaluate

results = json.load(open(RESULTS_FILE))
preds = [r['generated_report'] for r in results]
refs  = [r['ground_truth']     for r in results]

bleu  = evaluate.load('sacrebleu')
rouge = evaluate.load('rouge')

b = bleu.compute(predictions=preds,  references=[[r] for r in refs])
ro = rouge.compute(predictions=preds, references=refs)

print(f"BLEU-4   : {b['score']:.2f}")
print(f"ROUGE-1  : {ro['rouge1']:.4f}")
print(f"ROUGE-2  : {ro['rouge2']:.4f}")
print(f"ROUGE-L  : {ro['rougeL']:.4f}")

try:
    from radgraph import F1RadGraph
    f1rg = F1RadGraph(reward_level='partial')
    mean_f1, *_ = f1rg(hypothesis_reports=preds, reference_reports=refs)
    print(f"RadGraph F1 (partial): {mean_f1:.4f}")
except Exception as e:
    print(f'RadGraph F1 skipped: {e}')

## 11 · Save outputs

In [ ]:
import shutil, os

OUT = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

for src, name in [
    (RESULTS_FILE,   'results_medgemma.json'),
    (RETRIEVAL_OUT,  'output_rag_radgraph.json'),
    (ALIGN_RADGRAPH, 'radiology_report_radgraph.json'),
]:
    if os.path.isfile(src):
        shutil.copy(src, f'{OUT}/{name}')

# LoRA checkpoint (adapter weights only, ~200 MB)
if os.path.isdir(CHECKPOINT_DIR):
    shutil.copytree(CHECKPOINT_DIR, f'{OUT}/medgemma_dpo', dirs_exist_ok=True)

print('Saved outputs:')
for f in sorted(os.listdir(OUT)):
    size = os.path.getsize(f'{OUT}/{f}') if os.path.isfile(f'{OUT}/{f}') else 0
    print(f'  {f}  ({size:,} bytes)' if size else f'  {f}/')